# Ranking predictions - pairwise learning-to-rank

The notebook `ranking_predictions.ipynb` regresses each team's season points, then sorts. Here we use a different
idea that matches the actual objective (an **ordering**) better:

1. For every pair of teams of the same conference and season, learn **P(team A finishes ahead of team B)** from the
   *difference* of their pre-season features (RankNet / Bradley-Terry style).
2. A team's score = its average win probability against every other team of its conference
   (a Copeland / Borda aggregation). Sorting the scores gives the ranking.

Why it can help:
* The model only has to learn *relative* strength inside one conference, so season-wide effects (era, number of games) drop out.
* A conference of 15 teams gives 105 pairs per season: many more training examples than 15 regression rows, which
  matters because the data is small.
* Training pairs are antisymmetric by construction (A vs B and B vs A), which removes a source of bias.

Models compared (same walk-forward protocol and metrics as `ranking_predictions.ipynb`):
* **Pairwise logistic** - linear, regularised.
* **Pairwise boosting** - shallow gradient-boosting classifier on feature differences (non-linear).
* **Blend** - average of the pairwise logistic, pairwise boosting and the Ridge regression of the other notebook.
* **Ridge (reference)** and the **naive baselines**.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.ensemble import HistGradientBoostingClassifier

import ranking_utils as ru
RS = ru.RANDOM_STATE

## 1. Data (same preprocessing as the other notebook)

In [ ]:
df = ru.build_dataset(ru.load_teams())
groups = ru.feature_groups(df)

selection_file = Path("ranking_selected_features.json")
FEATURES = json.loads(selection_file.read_text())["features"] if selection_file.exists() else ru.flatten(groups)
print("Features:", FEATURES)

EVAL_YEARS = sorted(int(y) for y in df.loc[df["year"] >= ru.TEST_START_YEAR, "year"].unique())
print(f"Evaluation seasons: {EVAL_YEARS}")
assert not df[FEATURES].isna().any().any()

## 2. Pairwise ranker

`PairwiseRanker.fit` builds, for each (season, conference) of the training seasons, all pairs (better team, worse team)
and learns on the **feature difference** `x_better - x_worse` (label 1) and its mirror `x_worse - x_better` (label 0).
At prediction time every ordered pair of the conference is scored and the probabilities are symmetrised
(`(p(a,b) + 1 - p(b,a)) / 2`) so that the result is exactly antisymmetric.

In [ ]:
def pair_differences(group, features):
    """Feature differences (better team - worse team) for all unordered pairs of a conference-season."""
    X = group[features].to_numpy(float)
    rank = group["conf_rank"].to_numpy()
    i, j = np.triu_indices(len(group), k=1)
    better_first = rank[i] < rank[j]
    better = np.where(better_first, i, j)
    worse = np.where(better_first, j, i)
    return X[better] - X[worse]


class PairwiseRanker:
    def __init__(self, make_classifier):
        self.make_classifier = make_classifier

    def fit(self, train, features):
        self.features = features
        diffs = [pair_differences(g, features)
                 for (_, conf), g in train.groupby(["year", "confID"])
                 if conf != "NONE" and len(g) >= 2]
        D = np.vstack(diffs)
        self.n_pairs = len(D)
        X = np.vstack([D, -D])
        y = np.r_[np.ones(len(D)), np.zeros(len(D))]
        self.clf = self.make_classifier().fit(X, y)
        return self

    def score(self, test):
        scores = pd.Series(0.0, index=test.index)
        for _, g in test.groupby(["year", "confID"]):
            n = len(g)
            if n < 2:
                continue
            X = g[self.features].to_numpy(float)
            i, j = np.where(~np.eye(n, dtype=bool))           # all ordered pairs
            d = X[i] - X[j]
            p = (self.clf.predict_proba(d)[:, 1] + 1 - self.clf.predict_proba(-d)[:, 1]) / 2
            scores.loc[g.index] = np.bincount(i, weights=p, minlength=n) / (n - 1)
        return scores


def walk_forward_pairwise(df, make_classifier, features, eval_years):
    outputs = []
    for year in eval_years:
        train = df[(df["year"] < year) & (df["n_hist"] >= 1)]
        test = df[df["year"] == year]
        if test.empty:
            continue
        ranker = PairwiseRanker(make_classifier).fit(train, features)
        out = test[["year", "tmID", "confID"]].copy()
        out["score"] = ranker.score(test)
        outputs.append(out)
    return pd.concat(outputs, ignore_index=True)


pairwise_models = {
    # linear, no intercept needed (differences are centred); strong regularisation
    "Pairwise logistic": lambda: make_pipeline(StandardScaler(with_mean=False),
                                              LogisticRegression(C=0.1, fit_intercept=False, max_iter=2000)),
    # non-linear on the differences
    "Pairwise boosting": lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150,
                                                                min_samples_leaf=50, l2_regularization=1.0,
                                                                random_state=RS),
}
ridge_factory = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))

## 3. Walk-forward evaluation (train on seasons < Y, predict Y)

In [ ]:
preds = {
    "Naive: previous season": ru.baseline_scores(df, "pts_z_lag1", EVAL_YEARS),
    "Naive: mean 3 seasons": ru.baseline_scores(df, "pts_z_mean3", EVAL_YEARS),
    "Ridge (reference)": ru.walk_forward(df, ridge_factory, FEATURES, EVAL_YEARS),
}
for name, factory in pairwise_models.items():
    preds[name] = walk_forward_pairwise(df, factory, FEATURES, EVAL_YEARS)

# Blend: average of the within-conference z-scores of three scores.
def within_group_z(pred):
    s = pred.set_index(["year", "tmID"])["score"]
    g = pred.set_index(["year", "tmID"])[["confID"]].join(s)
    grouped = g.groupby(["year", "confID"])["score"]
    return ((g["score"] - grouped.transform("mean")) / grouped.transform("std").replace(0, np.nan)).fillna(0.0)

blend_parts = ["Ridge (reference)", "Pairwise logistic", "Pairwise boosting"]
blend = preds["Ridge (reference)"].set_index(["year", "tmID"])[["confID"]].copy()
blend["score"] = sum(within_group_z(preds[p]).reindex(blend.index) for p in blend_parts) / len(blend_parts)
preds["Blend (ridge + pairwise)"] = blend.reset_index()

ranked = {name: ru.rank_predictions(p, df) for name, p in preds.items()}
season_tables = {name: ru.season_metrics(r) for name, r in ranked.items()}

def pair_accuracy(r):
    taus = [kendalltau(g["pred_rank"], g["conf_rank"])[0] for _, g in r.groupby(["year", "confID"])]
    return (1 + np.mean(taus)) / 2      # share of team pairs ordered correctly

summary = pd.DataFrame({name: ru.summarise(t) for name, t in season_tables.items()}).T
summary["pair_accuracy"] = pd.Series({name: pair_accuracy(r) for name, r in ranked.items()})
summary["mae_std_over_seasons"] = pd.Series({name: t["mae"].std() for name, t in season_tables.items()})
summary = summary.sort_values("mae")

n_typical = int(round(ranked["Ridge (reference)"]["conf_size"].mean()))
print(f"{ranked['Ridge (reference)'].groupby(['year','confID']).ngroups} conference rankings evaluated | "
      f"random reference: MAE {ru.random_ranking_mae(n_typical):.2f}, top-{ru.PLAYOFF_SPOTS} overlap {ru.PLAYOFF_SPOTS/n_typical:.2f}, pair accuracy 0.50")
display(summary.round(3))

In [ ]:
order = list(summary.index)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, metric, better in zip(axes, ["mae", "spearman", "pair_accuracy"], ["lower", "higher", "higher"]):
    values = summary.loc[order, metric]
    colors = ["#DD8452" if n.startswith("Naive") else "#55A868" if n.startswith(("Pairwise", "Blend")) else "#4C72B0" for n in order]
    ax.barh(order[::-1], values.values[::-1], color=colors[::-1])
    for y, v in enumerate(values.values[::-1]):
        ax.text(v, y, f" {v:.3f}", va="center", fontsize=8)
    ax.set_title(f"{metric} ({better} is better)"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
for n, t in season_tables.items():
    s = t.groupby("year")["mae"].mean()
    ax.plot(s.index, s.values, marker="o", label=n, ls="--" if n.startswith("Naive") else "-")
ax.set_xlabel("Season"); ax.set_ylabel("MAE (rank positions)"); ax.set_title("Error per test season")
ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 4. Is the gain real?

Paired bootstrap over (season, conference) rankings (2000 draws). Positive = the model has a lower MAE than the
comparison model; "significant" means the 95% interval excludes 0. We compare against the naive model **and** against
the Ridge regression, since the real question is whether the pairwise formulation adds something.

In [ ]:
key = ["year", "tmID"]
base_index = ranked["Naive: previous season"].set_index(key).sort_index().index
group_id = (ranked["Naive: previous season"].set_index(key).sort_index()["confID"] + "-"
            + base_index.get_level_values("year").astype(str)).to_numpy()
abs_err = {n: r.set_index(key).sort_index()["abs_err"].reindex(base_index).to_numpy() for n, r in ranked.items()}

rows = []
for name in ranked:
    row = {"model": name}
    for ref_name, ref in [("naive", "Naive: previous season"), ("ridge", "Ridge (reference)")]:
        if name == ref:
            continue
        gain, (lo, hi) = ru.paired_bootstrap(abs_err[ref], abs_err[name], group_id)
        row.update({f"gain_vs_{ref_name}": gain, f"ci_low_{ref_name}": lo, f"ci_high_{ref_name}": hi})
    rows.append(row)
significance = pd.DataFrame(rows).set_index("model")
display(significance.loc[summary.index].round(3))

## 5. Verdict

In [ ]:
new_models = [n for n in summary.index if n.startswith(("Pairwise", "Blend"))]
best_new = summary.loc[new_models].sort_values(["mae", "spearman"], ascending=[True, False]).index[0]
overall_best = summary.drop(index=[n for n in summary.index if n.startswith("Naive")]).index[0]
random_mae = ru.random_ranking_mae(n_typical)

def line(name):
    s, g = summary.loc[name], significance.loc[name]
    return (f"{name}: MAE {s['mae']:.2f} | Spearman {s['spearman']:.2f} | pair accuracy {s['pair_accuracy']:.1%} | "
            f"top-8 overlap {s['top_overlap']:.0%}")

print("Best new model :", line(best_new))
print("Ridge reference:", line("Ridge (reference)"))
print("Naive          :", line("Naive: previous season"))
g = significance.loc[best_new]
print(f"\n{best_new} vs naive: {g['gain_vs_naive']:+.2f} positions, 95% CI [{g['ci_low_naive']:+.2f}, {g['ci_high_naive']:+.2f}]")
if best_new != "Ridge (reference)":
    print(f"{best_new} vs Ridge: {g['gain_vs_ridge']:+.2f} positions, 95% CI [{g['ci_low_ridge']:+.2f}, {g['ci_high_ridge']:+.2f}]")
print(f"Skill vs random ranking: {1 - summary.loc[best_new, 'mae'] / random_mae:.0%} of the random error removed")

print("\nBest model overall in this notebook:", overall_best)
if best_new != "Ridge (reference)" and g["ci_low_ridge"] > 0:
    print("-> The pairwise formulation brings a statistically reliable gain over Ridge.")
elif best_new != "Ridge (reference)" and g["gain_vs_ridge"] > 0:
    print("-> The pairwise formulation is slightly ahead of Ridge, but the difference is within noise on these seasons.")
else:
    print("-> The pairwise formulation does not beat Ridge here: the signal is limited by luck and parity, not by the model class.")

In [ ]:
best_ranked = ranked[overall_best]
last_year = EVAL_YEARS[-1]
for conf, g in best_ranked[best_ranked["year"] == last_year].groupby("confID"):
    print(f"\nSeason {last_year} - conference {conf} - {overall_best}")
    display(g.sort_values("pred_rank")[["pred_rank", "tmID", "conf_rank", "err", "score"]].rename(
        columns={"conf_rank": "actual_rank"}).round(3))

best_ranked.to_csv("ranking_predictions_pairwise_output.csv", index=False)
print("\nSaved -> ranking_predictions_pairwise_output.csv")

## Notes
* Training uses every pair inside a conference-season, so the "effective" sample is a few thousand pairs even though there are
  only ~30 teams per season. Pairs of the same season are correlated, which is why evaluation is always done on whole
  rankings and the bootstrap resamples whole rankings.
* `pair_accuracy` is the fraction of team pairs ordered correctly (Kendall tau rescaled to [0, 1]); 0.5 is chance.
* Hyper-parameters (`C`, tree depth, ...) are fixed and conservative on purpose: tuning on a handful of test seasons would
  mostly fit noise.